In [2]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv('cdc_brfss.csv')
print(data.shape)
data.head()

(253680, 22)


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [3]:
# Drop NaN value
print("Any null value:", any(data.isnull()))
print("Any NaN value:", any(data.isna()))
print("Before Droping NaN Number of Rows:", len(data))

data = data.dropna()
data = data.drop_duplicates()
print("After Droping NaN Number of Rows and Duplicates:", len(data))

# Move 'Diabetes_binary' column to the end of dataframe
data['Diabetes_binary'] = data.pop('Diabetes_binary')

data.tail()

Any null value: True
Any NaN value: True
Before Droping NaN Number of Rows: 253680
After Droping NaN Number of Rows and Duplicates: 229474


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
253675,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0,0.0
253676,1.0,1.0,1.0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0,1.0
253677,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0,0.0
253678,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0,0.0
253679,1.0,1.0,1.0,25.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,0.0,9.0,6.0,2.0,1.0


In [3]:
cdc_corr = data.corr()
print(cdc_corr.iloc[-1,:].sort_values(ascending=False))

Diabetes_binary         1.000000
GenHlth                 0.276940
HighBP                  0.254318
DiffWalk                0.205302
BMI                     0.205086
HighChol                0.194944
Age                     0.177263
HeartDiseaseorAttack    0.168213
PhysHlth                0.156211
Stroke                  0.099193
CholCheck               0.072523
MentHlth                0.054153
Smoker                  0.045504
Sex                     0.032724
AnyHealthcare           0.025331
NoDocbcCost             0.020048
Fruits                 -0.024805
Veggies                -0.041734
HvyAlcoholConsump      -0.065950
PhysActivity           -0.100404
Education              -0.102686
Income                 -0.140659
Name: Diabetes_binary, dtype: float64


In [4]:
#split

from sklearn.model_selection import train_test_split

def data_split(data):
    
    X = data.iloc[:,:-1]
    y = data.iloc[:,-1]
    
    train_x, temp_x, train_y, temp_y = train_test_split(X,y,test_size = 0.3, random_state=0)

    test_x, val_x, test_y, val_y = train_test_split(temp_x, temp_y, test_size=0.3333, random_state=0, stratify=temp_y)


# Check sizes
    print("Train size:", len(train_x))
    print("Test size:", len(test_x))
    print("Validation size:", len(val_x))
    
    return train_x, test_x, train_y, test_y, val_x, val_y

train_x, test_x, train_y, test_y, val_x, val_y = data_split(data)

Train size: 160631
Test size: 45897
Validation size: 22946


In [5]:

train_df = pd.concat([train_x, train_y], axis=1)
test_df  = pd.concat([test_x, test_y], axis=1)
val_df   = pd.concat([val_x, val_y], axis=1)

train_df.to_csv("cdc_train.csv", index=False)
test_df.to_csv("cdc_test.csv", index=False)
val_df.to_csv("cdc_validation.csv", index=False)


In [7]:
# extratrees
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score

model = ExtraTreesClassifier(random_state = 0)

model.fit(train_x, train_y)

print(cross_val_score(model, train_x, train_y, scoring = 'accuracy', cv = 5, n_jobs = 1).mean())

print(cross_val_score(model, train_x, train_y, scoring = 'roc_auc', cv = 5, n_jobs = 1).mean())


0.8363329696073045
0.7536380184670077


In [ ]:
#catmod

from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

cat_mod = CatBoostClassifier(
    iterations=100,      
    learning_rate=0.1,   
    depth=6,              
    verbose=0             
)

cat_mod.fit(train_x, train_y)
y_pred = cat_mod.predict(test_x)
accuracy = accuracy_score(test_y, y_pred)
print("Accuracy:", accuracy_score(test_y, y_pred))

auc_scores = cross_val_score(
    cat_mod,
    train_x,
    train_y,
    scoring='roc_auc',
    cv=5
)

print("Mean CV AUC:", auc_scores.mean())
print("All fold AUCs:", auc_scores)

In [ ]:
#adaboost
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=50,
                         learning_rate=1)

ada_model = ada.fit(train_x, train_y)

y_pred = ada_model.predict(test_x)

print("Accuracy:", accuracy_score(test_y, y_pred))

auc_scores = cross_val_score(
    ada,
    train_x,
    train_y,
    scoring='roc_auc',
    cv=5
)

print("Mean CV AUC:", auc_scores.mean())

In [ ]:
from c50py import C5Classifier
from sklearn.model_selection import StratifiedKFold
import numpy as np

c5 = C5Classifier(trials=1)

c5_model = c5.fit(train_x, train_y)

y_pred = c5_model.predict(test_x)

print("Accuracy:", accuracy_score(test_y, y_pred))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

auc_scores = []

for train_idx, val_idx in cv.split(train_x, train_y):
    X_train_fold = train_x.iloc[train_idx]
    y_train_fold = train_y.iloc[train_idx]
    X_val_fold = train_x.iloc[val_idx]
    y_val_fold = train_y.iloc[val_idx]

    c5 = C5Classifier(trials=1)
    c5.fit(X_train_fold, y_train_fold)

    y_proba = c5.predict_proba(X_val_fold)[:, 1]
    auc = roc_auc_score(y_val_fold, y_proba)
    auc_scores.append(auc)

print("Mean CV AUC:", np.mean(auc_scores))

In [ ]:
# import c50py
# print(c50py)
# print(dir(c50py))

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate

xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    eval_metric='logloss',   # avoids warning
    use_label_encoder=False,
    random_state=0
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

scores = cross_validate(
    xgb,
    train_x,
    train_y,
    scoring=['roc_auc', 'accuracy'],
    cv=cv
)

print("XGBoost Mean CV AUC:", scores['test_roc_auc'].mean())
print("XGBoost Mean CV Accuracy:", scores['test_accuracy'].mean())

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

scoring = ['roc_auc', 'accuracy']

knn_scores = cross_validate(
    knn_pipeline,
    train_x,
    train_y,
    scoring=scoring,
    cv=cv
)

print("KNN Mean CV AUC:", knn_scores['test_roc_auc'].mean())
print("KNN Mean CV Accuracy:", knn_scores['test_accuracy'].mean())

In [ ]:
#experimental shap

# ================================
# BPNN (MLP) + SHAP (ROBUST FIX)
# ================================

import shap
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Build model
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        max_iter=500,
        random_state=0
    ))
])

# Fit
mlp_pipeline.fit(train_x, train_y)

# Background sample
background = shap.sample(train_x, 100, random_state=0)

# Wrapped predict function
predict_fn = lambda x: mlp_pipeline.predict_proba(
    pd.DataFrame(x, columns=train_x.columns)
)

explainer = shap.KernelExplainer(predict_fn, background)

# Compute SHAP on subset
sample_data = train_x.sample(200, random_state=0)
shap_values = explainer.shap_values(sample_data)

# Handle different SHAP output formats safely
if isinstance(shap_values, list):
    shap_class1 = shap_values[1]
else:
    shap_class1 = shap_values[:, :, 1]  # for 3D output

# Plot summary
shap.summary_plot(shap_class1, sample_data)

# Numerical importance
mean_shap = np.abs(shap_class1).mean(axis=0)

importance_df = pd.DataFrame({
    'Feature': train_x.columns,
    'SHAP Importance': mean_shap
}).sort_values(by='SHAP Importance', ascending=False)

print("\nTop 10 Most Important Covariates:")
print(importance_df.head(10))

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

# 1️⃣ Use SHAP importance from BPNN
top_features = importance_df['Feature'].head(7).tolist()  # top 10 features

# 2️⃣ Fit ExtraTrees using only top features
et = ExtraTreesClassifier(n_estimators=200, random_state=0)
et.fit(train_x[top_features], train_y)

# 3️⃣ Evaluate
from sklearn.model_selection import cross_val_score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
et_auc = cross_val_score(et, train_x[top_features], train_y, scoring='roc_auc', cv=cv)
et_acc = cross_val_score(et, train_x[top_features], train_y, scoring='accuracy', cv=cv)

print("ExtraTrees Mean CV AUC:", et_auc.mean())
print("ExtraTrees Mean CV Accuracy:", et_acc.mean())

In [5]:
# testing a novel approach

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from econml.dml import CausalForestDML

def data_split(data):
    X = data.iloc[:,:-1]
    y = data.iloc[:,-1]
    
    train_x, temp_x, train_y, temp_y = train_test_split(
        X, y, test_size=0.3, random_state=0
    )

    test_x, val_x, test_y, val_y = train_test_split(
        temp_x, temp_y, test_size=0.3333, random_state=0, stratify=temp_y
    )

    print("Train size:", len(train_x))
    print("Test size:", len(test_x))
    print("Validation size:", len(val_x))
    
    return train_x, test_x, train_y, test_y, val_x, val_y

train_x, test_x, train_y, test_y, val_x, val_y = data_split(data)

if "BMI_binary" not in train_x.columns:
    train_x["BMI_binary"] = (train_x["BMI"] >= 30).astype(int)
    test_x["BMI_binary"] = (test_x["BMI"] >= 30).astype(int)
    val_x["BMI_binary"] = (val_x["BMI"] >= 30).astype(int)

treatment_cols = ["PhysActivity", "BMI_binary", "Smoker"]

T_train = train_x[treatment_cols].values
T_test  = test_x[treatment_cols].values
T_val   = val_x[treatment_cols].values

X_train = train_x.drop(columns=treatment_cols)
X_test  = test_x.drop(columns=treatment_cols)
X_val   = val_x.drop(columns=treatment_cols)

Y_train = train_y.values
Y_test  = test_y.values
Y_val   = val_y.values

model_y = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=10,
    random_state=0
)

model_t = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=10,
    random_state=0
)

causal_model = CausalForestDML(
    model_y=model_y,
    model_t=model_t,
    n_estimators=300,
    min_samples_leaf=20,
    max_depth=10,
    random_state=0
)

causal_model.fit(Y_train, T_train, X=X_train)

n = len(X_test)

T_unhealthy = np.array([[0, 1, 1]] * n)
T_exercise  = np.array([[1, 1, 1]] * n)
T_weight    = np.array([[0, 0, 1]] * n)
T_smoking   = np.array([[0, 1, 0]] * n)
T_two       = np.array([[1, 0, 1]] * n)
T_all       = np.array([[1, 0, 0]] * n)

effect_exercise = causal_model.effect(X_test, T0=T_unhealthy, T1=T_exercise)
effect_weight   = causal_model.effect(X_test, T0=T_unhealthy, T1=T_weight)
effect_smoking  = causal_model.effect(X_test, T0=T_unhealthy, T1=T_smoking)
effect_two      = causal_model.effect(X_test, T0=T_unhealthy, T1=T_two)
effect_all      = causal_model.effect(X_test, T0=T_unhealthy, T1=T_all)

def find_best_intervention(i):
    effects = {
        "Exercise only": effect_exercise[i],
        "Weight loss only": effect_weight[i],
        "Quit smoking": effect_smoking[i],
        "Exercise + weight loss": effect_two[i],
        "All three": effect_all[i]
    }
    
    best = min(effects, key=effects.get)
    return best, effects[best]

results = [find_best_intervention(i) for i in range(len(X_test))]

for i in range(10):
    print(f"\nPerson {i}")
    print(f"Best intervention: {results[i][0]}")
    print(f"Estimated risk reduction: {abs(results[i][1]):.4f}")

best_choices = [r[0] for r in results]
print("\nIntervention distribution:")
print(pd.Series(best_choices).value_counts())

print("\nAverage effects:")
print("Exercise:", effect_exercise.mean())
print("Weight loss:", effect_weight.mean())
print("Quit smoking:", effect_smoking.mean())
print("Exercise + weight:", effect_two.mean())
print("All three:", effect_all.mean())

val_effect = causal_model.effect(
    X_val,
    T0=np.array([[0,1,1]] * len(X_val)),
    T1=np.array([[1,0,0]] * len(X_val))
)

print("\nValidation effect (all interventions):", val_effect.mean())

Train size: 160631
Test size: 45897
Validation size: 22946

Person 0
Best intervention: Weight loss only
Estimated risk reduction: 0.0000

Person 1
Best intervention: Weight loss only
Estimated risk reduction: 0.0000

Person 2
Best intervention: Weight loss only
Estimated risk reduction: 0.0000

Person 3
Best intervention: Weight loss only
Estimated risk reduction: 0.0000

Person 4
Best intervention: Weight loss only
Estimated risk reduction: 0.0000

Person 5
Best intervention: All three
Estimated risk reduction: 0.0050

Person 6
Best intervention: Weight loss only
Estimated risk reduction: 0.0000

Person 7
Best intervention: Weight loss only
Estimated risk reduction: 0.0000

Person 8
Best intervention: Weight loss only
Estimated risk reduction: 0.0000

Person 9
Best intervention: Weight loss only
Estimated risk reduction: 0.0000

Intervention distribution:
Weight loss only          24586
Exercise only              7154
Exercise + weight loss     4986
Quit smoking               4963
Al

In [ ]:
# test 2
from sklearn.linear_model import LogisticRegression

if "BMI_binary" not in train_x.columns:
    train_x["BMI_binary"] = (train_x["BMI"] >= 30).astype(int)
    test_x["BMI_binary"] = (test_x["BMI"] >= 30).astype(int)
    val_x["BMI_binary"] = (val_x["BMI"] >= 30).astype(int)

for col in ["PhysActivity", "BMI_binary", "Smoker"]:
    train_x[col] = train_x[col].astype(int)
    test_x[col] = test_x[col].astype(int)
    val_x[col] = val_x[col].astype(int)

treatment_cols = ["PhysActivity", "BMI_binary", "Smoker"]
T_train = train_x[treatment_cols].values
T_test  = test_x[treatment_cols].values
T_val   = val_x[treatment_cols].values

X_train = train_x.drop(columns=treatment_cols)
X_test  = test_x.drop(columns=treatment_cols)
X_val   = val_x.drop(columns=treatment_cols)

Y_train = train_y.values
Y_test  = test_y.values
Y_val   = val_y.values

model_y = RandomForestRegressor(n_estimators=200, min_samples_leaf=10, random_state=0)
model_t = RandomForestRegressor(n_estimators=200, min_samples_leaf=10, random_state=0)

causal_model = CausalForestDML(
    model_y=model_y,
    model_t=model_t,
    n_estimators=300,
    min_samples_leaf=20,
    max_depth=10,
    random_state=0
)

causal_model.fit(Y_train, T_train, X=X_train)

n = len(X_test)
T_unhealthy = np.array([[0, 1, 1]] * n)
T_exercise  = np.array([[1, 1, 1]] * n)
T_weight    = np.array([[0, 0, 1]] * n)
T_smoking   = np.array([[0, 1, 0]] * n)
T_two       = np.array([[1, 0, 1]] * n)
T_all       = np.array([[1, 0, 0]] * n)

effect_exercise = causal_model.effect(X_test, T0=T_unhealthy, T1=T_exercise)
effect_weight   = causal_model.effect(X_test, T0=T_unhealthy, T1=T_weight)
effect_smoking  = causal_model.effect(X_test, T0=T_unhealthy, T1=T_smoking)
effect_two      = causal_model.effect(X_test, T0=T_unhealthy, T1=T_two)
effect_all      = causal_model.effect(X_test, T0=T_unhealthy, T1=T_all)

def find_best_intervention(i):
    effects = {
        "Exercise only": effect_exercise[i],
        "Weight loss only": effect_weight[i],
        "Quit smoking": effect_smoking[i],
        "Exercise + weight loss": effect_two[i],
        "All three": effect_all[i]
    }
    best = min(effects, key=effects.get)
    return best, effects[best]

results = [find_best_intervention(i) for i in range(len(X_test))]

for i in range(10):
    print(f"\nPerson {i}")
    print(f"Best intervention: {results[i][0]}")
    print(f"Estimated risk reduction: {abs(results[i][1]):.4f}")

best_choices = [r[0] for r in results]
print("\nIntervention distribution:")
print(pd.Series(best_choices).value_counts())

print("\nAverage effects:")
print("Exercise:", effect_exercise.mean())
print("Weight loss:", effect_weight.mean())
print("Quit smoking:", effect_smoking.mean())
print("Exercise + weight:", effect_two.mean())
print("All three:", effect_all.mean())

val_effect = causal_model.effect(X_val, T0=np.array([[0,1,1]]*len(X_val)), T1=np.array([[1,0,0]]*len(X_val)))
print("\nValidation effect (all interventions):", val_effect.mean())